<a href="https://colab.research.google.com/github/Stepa555/chocolate-sales-dashboard/blob/main/_2022_2023.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Загрузка и первичная разведка

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Загружаем файл
from google.colab import files
uploaded = files.upload()


Saving Chocolate_Sales.csv to Chocolate_Sales.csv


In [ ]:
df = pd.read_csv('Chocolate_Sales.csv')

In [ ]:
# Разведка

print('=== Размер данных ===')
print(df.shape)

print('\n=== Первые 5 строк ===')
print(df.head())

print('\n=== Типы данных ===')
print(df.dtypes)

print('\n=== Пропуски ===')
print(df.isnull().sum())

print('\n=== Дубликаты ===')
print(f'Дубликатов: {df.duplicated().sum()}')

print('\n=== Описательная статистика ===')
print(df.describe())

=== Размер данных ===
(200000, 11)

=== Первые 5 строк ===
     Order_ID            Product    Country Channel    Salesperson  \
0  ORD-069833   Truffle Gift Box  Australia  Retail    Arjun Mehta   
1  ORD-090726       85% Dark Bar  Australia  Retail    Arjun Mehta   
2  ORD-042159       70% Dark Bar      Japan  Retail  Hannah Müller   
3  ORD-197166  Hazelnut Milk Bar    Germany  Retail    Arjun Mehta   
4  ORD-112162  Almond Crunch Bar  Australia  Retail      Yuki Sato   

   Order_Date  Discount_Pct  Price_per_Box  Marketing_Spend  Boxes_Shipped  \
0  2022-12-11           3.5          13.72           202.03             71   
1  2023-03-14           9.4           3.30            55.18             84   
2  2023-12-21           4.9          18.21            60.65             35   
3  2023-12-18          15.0           2.66            52.00             92   
4  2023-08-18           4.4           2.75           187.44            214   

   Amount  
0  912.31  
1  245.91  
2   583.7  
3  

# Очистка данных

In [ ]:
# Создаем копию чтобы не потерять оригинал
df_clean = df.copy()

# Приводим Order_Date к типу datetime
df_clean['Order_Date'] = pd.to_datetime(df_clean['Order_Date'], errors='coerce')

# Преобразуем Amount в число (убираем лишние символы, если есть)
df_clean['Amount'] = pd.to_numeric(df_clean['Amount'], errors='coerce')

# Удаляем строки с отрицательным количеством коробок (невозможно!)
df_clean = df_clean[df_clean['Boxes_Shipped'] > 0]

# Удаляем выбросы: например, Boxes_Shipped > 1000 (редкий экстремальный случай).
Q1 = df_clean['Boxes_Shipped'].quantile(0.25)
Q3 = df_clean['Boxes_Shipped'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df_clean = df_clean[(df_clean['Boxes_Shipped'] >= lower_bound) & (df_clean['Boxes_Shipped'] <= upper_bound)]

# Заполняем пропуски
# Discount_Pct: если скидка не указана, ставим 0
df_clean['Discount_Pct'].fillna(0, inplace=True)

# Price_per_Box: заполняем медианой по каждому продукту (более точный способ)
df_clean['Price_per_Box'] = df_clean.groupby('Product')['Price_per_Box'].transform(
    lambda x: x.fillna(x.median())
)

# Marketing_Spend: заполняем медианой по каналу продаж
df_clean['Marketing_Spend'] = df_clean.groupby('Channel')['Marketing_Spend'].transform(
    lambda x: x.fillna(x.median())
)

# Order_Date: если дата пропущена, удаляем строку (их мало, ~437 из 200k)
df_clean = df_clean.dropna(subset=['Order_Date'])

# Создаем новые полезные признаки
df_clean['Год'] = df_clean['Order_Date'].dt.year
df_clean['Месяц'] = df_clean['Order_Date'].dt.month
df_clean['День_недели'] = df_clean['Order_Date'].dt.dayofweek  # 0=пн, 6=вс
df_clean['Квартал'] = df_clean['Order_Date'].dt.quarter

# Проверяем, что получилось
print("=== После очистки ===")
print(f"Размер: {df_clean.shape}")
print(f"Пропусков:\n{df_clean.isnull().sum()}")
print(df_clean.head())

/tmp/ipykernel_2142/992566453.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['Discount_Pct'].fillna(0, inplace=True)


=== После очистки ===
Размер: (185172, 15)
Пропусков:
Order_ID              0
Product               0
Country               0
Channel               0
Salesperson           0
Order_Date            0
Discount_Pct          0
Price_per_Box         0
Marketing_Spend       0
Boxes_Shipped         0
Amount             2536
Год                   0
Месяц                 0
День_недели           0
Квартал               0
dtype: int64
     Order_ID            Product    Country Channel    Salesperson Order_Date  \
0  ORD-069833   Truffle Gift Box  Australia  Retail    Arjun Mehta 2022-12-11   
1  ORD-090726       85% Dark Bar  Australia  Retail    Arjun Mehta 2023-03-14   
2  ORD-042159       70% Dark Bar      Japan  Retail  Hannah Müller 2023-12-21   
3  ORD-197166  Hazelnut Milk Bar    Germany  Retail    Arjun Mehta 2023-12-18   
4  ORD-112162  Almond Crunch Bar  Australia  Retail      Yuki Sato 2023-08-18   

   Discount_Pct  Price_per_Box  Marketing_Spend  Boxes_Shipped  Amount   Год  \
0     

In [ ]:
# Найди строки, где Amount стал NaN после преобразования
bad_amounts = df_clean[df_clean['Amount'].isnull()]

# Посмотри на них
print(bad_amounts[['Order_ID', 'Amount']].head(20))

# Или посмотри уникальные «плохие» значения
print(df['Amount'].unique()[:50])  # Смотрим на оригинальный столбец

        Order_ID  Amount
12    ORD-185273     NaN
157   ORD-125198     NaN
391   ORD-186735     NaN
467   ORD-145804     NaN
506   ORD-007560     NaN
645   ORD-031775     NaN
865   ORD-081106     NaN
923   ORD-086404     NaN
991   ORD-099771     NaN
1116  ORD-145854     NaN
1158  ORD-048810     NaN
1175  ORD-091783     NaN
1210  ORD-039909     NaN
1218  ORD-084997     NaN
1272  ORD-196951     NaN
1302  ORD-068722     NaN
1371  ORD-010502     NaN
1374  ORD-133405     NaN
1414  ORD-139547     NaN
1444  ORD-083558     NaN
['912.31' '245.91' '583.7' '211.27' '549.69' '527.01' '470.63' '629.51'
 '181.16' '588.18' '299.92' '440.21' '$383.66' '762.63' '614.78' '447.35'
 '1545.89' '336.09' '716.19' '607.38' '709.31' '265.65' '208.23' '271.52'
 '186.04' '411.12' '356.92' '281.27' '485.08' '303.44' '727.09' '355.95'
 '532.87' '513.52' '331.89' '427.33' '1310.5' '223.45' '853.17' '222.92'
 '273.01' '522.06' '549.42' '307.83' '553.27' '540.86' '501.71' '337.12'
 '1067.2' '414.13']


In [ ]:
# 1. Убираем символ $ и запятые из Amount
df_clean['Amount'] = df_clean['Amount'].astype(str).str.replace(r'[\$,]', '', regex=True)

# 2. Преобразуем в число
df_clean['Amount'] = pd.to_numeric(df_clean['Amount'], errors='coerce')

# 3. Пересчитываем Amount, если все еще есть пропуски
# Проверяем, остались ли пропуски
print(f"Пропусков в Amount: {df_clean['Amount'].isnull().sum()}")

# 4. Если пропуски остались, пересчитываем их через Boxes_Shipped * Price_per_Box
df_clean['Amount'] = df_clean['Amount'].fillna(df_clean['Boxes_Shipped'] * df_clean['Price_per_Box'])

# 5. Проверяем результат
print(f"Пропусков в Amount после пересчета: {df_clean['Amount'].isnull().sum()}")

Пропусков в Amount: 2536
Пропусков в Amount после пересчета: 0


In [ ]:
# 8. Сохраняем чистый файл для Power BI
df_clean.to_csv('chocolate_clean.csv', index=False)
files.download('chocolate_clean.csv')

print("✅ Данные очищены и готовы к загрузке в Power BI!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Данные очищены и готовы к загрузке в Power BI!


# SQL-запросы (в Colab через pandasql)

In [ ]:
!pip install pandasql
from pandasql import sqldf
pysqldf = lambda q: sqldf(q, globals())

  Preparing metadata (setup.py) ... done
  Created wheel for pandasql: filename=pandasql-0.7.3-py3-none-any.whl size=26773 sha256=3f5d3cc636e2c600f4e011975511b595b399ae3951aabf0734979ce1eabc2c91
  Stored in directory: /root/.cache/pip/wheels/15/a1/e7/6f92f295b5272ae5c02365e6b8fa19cb93f16a537090a1cf27
Successfully built pandasql


In [ ]:
# 1. Топ-5 продуктов по выручке

query1 = """
SELECT Product, SUM(Amount) as Total_Revenue, COUNT(*) as Orders
FROM df_clean
GROUP BY Product
ORDER BY Total_Revenue DESC
LIMIT 5
"""
print("=== Топ-5 продуктов ===")
print(pysqldf(query1))


=== Топ-5 продуктов ===
                Product  Total_Revenue  Orders
0          70% Dark Bar    24605526.56   62207
1      Truffle Gift Box    16624848.40   29561
2  Mixed Assortment Box    13157346.49   16499
3      Milk Classic Bar     5838681.92   15533
4     Pralines Gift Box     5297727.82   12127


In [ ]:
# 2. Анализ по каналам продаж (средний чек, выручка)

query2 = """
SELECT Channel,
       Count(*) as Orders,
       SUM(Amount) as Revenue,
       AVG(Amount) as Avg_Check,
       AVG(Discount_Pct) as Avg_Discount
From df_clean
Group By Channel
Order by Revenue DESC
"""
print("\n=== Анализ по каналам ===")
print(pysqldf(query2))


=== Анализ по каналам ===
     Channel  Orders      Revenue   Avg_Check  Avg_Discount
0     Retail  109585  47758086.38  435.808609     12.982961
1  Wholesale   52757  30056641.74  569.718554     12.825089
2     Online   22830  10038167.57  439.691965     12.969157


In [ ]:
# Сезонность по месяцам (линейный график в Power BI)

query3 = """
SELECT Месяц,
      COUNT(*) AS Orders,
      SUM(Amount) AS Revenue,
      AVG(Amount) AS Avg_Check
From df_clean
GROUP BY Месяц
ORDER BY Месяц
"""
print("\n=== Сезонность по месяцам ===")
print(pysqldf(query3))


=== Сезонность по месяцам ===
    Месяц  Orders     Revenue   Avg_Check
0       1   15485  8076572.30  521.573930
1       2   14440  6979496.25  483.344616
2       3   16053  7078256.90  440.930474
3       4   15335  6800092.38  443.436086
4       5   15841  7029187.11  443.733799
5       6   15373  6537186.51  425.238178
6       7   15814  6752443.19  426.991475
7       8   16139  7194034.01  445.754632
8       9   15350  7119921.26  463.838519
9      10   15764  7634649.95  484.309182
10     11   14799  7997254.99  540.391580
11     12   14779  8653800.84  585.547117


In [ ]:
# 4. Оконная функция: топ-3 продавца по выручке в каждой стране
query4 = """
SELECT Country, Salesperson, Revenue, Rank
FROM (
    SELECT Country, Salesperson,
           SUM(Amount) as Revenue,
           ROW_NUMBER() OVER (PARTITION BY Country ORDER BY SUM(Amount) DESC) as Rank
    FROM df_clean
    GROUP BY Country, Salesperson
)
WHERE Rank <= 3
ORDER BY Country, Rank
"""
print("\n=== Топ-3 продавца по странам ===")
print(pysqldf(query4))


=== Топ-3 продавца по странам ===
      Country   Salesperson      Revenue  Rank
0   Australia   Arjun Mehta  12865014.37     1
1   Australia  Priya Sharma   5614415.65     2
2   Australia  Emily Clarke   3351225.36     3
3      Brazil   Arjun Mehta   9707263.42     1
4      Brazil  Priya Sharma   4119834.47     2
5      Brazil  Emily Clarke   2488251.36     3
6     Germany   Arjun Mehta   3463699.79     1
7     Germany  Priya Sharma   1475085.60     2
8     Germany  Emily Clarke    920322.11     3
9       India   Arjun Mehta   2027571.83     1
10      India  Priya Sharma    887790.02     2
11      India  Emily Clarke    559812.31     3
12      Japan   Arjun Mehta   1687748.35     1
13      Japan  Priya Sharma    737631.24     2
14      Japan  Emily Clarke    471037.54     3
